In [2]:
from pathlib import Path
print('Python runtime ready')
print('Working directory:', Path.cwd())


Python runtime ready
Working directory: /content


In [3]:
%%writefile ecommerce_app.py
"""EduCommerce: in-memory application for practicing checkout and automated testing."""
from dataclasses import dataclass, field
from decimal import Decimal, ROUND_HALF_UP
from typing import Optional

D=lambda x: Decimal(str(x))

def money(x): return D(x).quantize(D('1'), rounding=ROUND_HALF_UP)

@dataclass
class Product:
    sku:str; title:str; price:Decimal; stock:int; active:bool=True
@dataclass
class Customer:
    customer_id:str; authenticated:bool=True; used_codes:set=field(default_factory=set)
@dataclass
class CartLine:
    sku:str; quantity:int
@dataclass
class Voucher:
    code:str; percent:int; min_subtotal:Decimal; cap:Decimal; active:bool=True
@dataclass
class Order:
    order_id:str; customer_id:str; total:Decimal; status:str; lines:list; discount:Decimal; shipping:Decimal; payment_method:str

class Catalog:
    def __init__(self):
        self.products={'A':Product('A','Wireless Mouse',D(200000),20),
                       'B':Product('B','Mechanical Keyboard',D(300000),15),
                       'C':Product('C','USB-C Hub',D(150000),12)}
    def get(self,sku):
        p=self.products.get(sku)
        if not p or not p.active: raise ValueError('PRODUCT_UNAVAILABLE')
        return p

class Cart:
    def __init__(self,catalog): self.catalog=catalog; self.lines=[]
    def add(self,sku,quantity):
        self.catalog.get(sku)
        if not isinstance(quantity,int) or quantity<1: raise ValueError('INVALID_QUANTITY')
        existing=next((x for x in self.lines if x.sku==sku),None)
        if existing: existing.quantity+=quantity
        else: self.lines.append(CartLine(sku,quantity))
    def subtotal(self): return sum((self.catalog.get(x.sku).price*x.quantity for x in self.lines),D(0))
    def total_quantity(self): return sum((line.quantity for line in self.lines),0)

class Inventory:
    def __init__(self,catalog): self.catalog=catalog
    def reserve(self,lines):
        for line in lines:
            if self.catalog.get(line.sku).stock < line.quantity: raise ValueError('OUT_OF_STOCK')
        for line in lines: self.catalog.get(line.sku).stock-=line.quantity
    def release(self,lines):
        for line in lines: self.catalog.get(line.sku).stock+=line.quantity

class Promotions:
    def __init__(self): self.codes={'SAVE10':Voucher('SAVE10',10,D(500000),D(100000))}
    def discount(self, code, subtotal, customer):
        if not code: return D(0)
        v=self.codes.get(code)
        if not v or not v.active: raise ValueError('INVALID_VOUCHER')
        if not customer.authenticated: raise ValueError('LOGIN_REQUIRED')
        if code in customer.used_codes: raise ValueError('VOUCHER_USED')
        if subtotal <= v.min_subtotal: raise ValueError('MIN_SUBTOTAL_NOT_MET')
        return min(money(subtotal*v.percent/D(100)),v.cap)


class Shipping:
    def calculate(self,subtotal,area):
        if area not in ('inner','outer'): raise ValueError('INVALID_AREA')
        return D(0) if area=='inner' and subtotal>=D(500000) else (D(30000) if area=='inner' else D(50000))

class Payments:
    def __init__(self): self.charges=[]
    def collect(self,amount,method,approved,key):
        if method=='COD': return 'PENDING_COD'
        if method!='ONLINE': raise ValueError('INVALID_PAYMENT_METHOD')
        if not approved: raise ValueError('PAYMENT_DECLINED')
        self.charges.append({'key':key,'amount':amount})
        return 'PAID'

class Checkout:
    def __init__(self, catalog=None):
        self.catalog = catalog or Catalog()
        self.inventory = Inventory(self.catalog)
        self.promotions = Promotions()
        self.shipping = Shipping()
        self.payments = Payments()
        self.orders = {}
        self.by_key = {}
        self.counter = 0

    def calculate_total(self, subtotal, discount, shipping):
        return max(D(0), subtotal - discount + shipping)

    def place(self, customer, cart, area='inner',
              voucher=None, method='COD',
              approved=True, key='REQ-001'):

        if not customer.authenticated:
            raise ValueError('LOGIN_REQUIRED')

        if not cart.lines:
            raise ValueError('EMPTY_CART')

        subtotal = cart.subtotal()
        discount = self.promotions.discount(voucher, subtotal, customer)
        shipping = self.shipping.calculate(subtotal, area)
        total = max(D(0), subtotal - discount + shipping)
        # INSTRUCTION BASELINE: reserve occurs before payment validation.
        self.inventory.reserve(cart.lines)
        if key in self.by_key:
            return self.orders[self.by_key[key]]
        # Intentionally preserve the source's lack of rollback for testing.
        payment_state = self.payments.collect(total, method, approved, key)

        # 5. Create order.
        self.counter += 1
        order = Order(
            f'ORD-{self.counter:04d}',
            customer.customer_id,
            total,
            payment_state,
            [(x.sku, x.quantity) for x in cart.lines],
            discount,
            shipping,
            method
        )

        # 6. Save order and idempotency key.
        self.orders[order.order_id] = order
        self.by_key[key] = order.order_id

        if voucher:
            customer.used_codes.add(voucher)

        return order

def fixture(fixture_name):
    service = Checkout()
    customer = Customer("U_TEST", authenticated=True)
    cart = Cart(service.catalog)

    if fixture_name == 'STANDARD':
        cart.add('A', 2)

    elif fixture_name == 'MULTI_ITEM':
        cart.add('A', 1)
        cart.add('B', 1)

    elif fixture_name == 'ANONYMOUS_USER':
        customer.authenticated = False
        cart.add('A', 1)

    elif fixture_name == 'EMPTY_CART':
        pass

    elif fixture_name == '500k':
        cart.add('A',1); cart.add('B',1)

    elif fixture_name == '499999':
        cart.add('A',1); cart.add('B',1)
        service.catalog.products['B'].price=D(299999)

    elif fixture_name == '600k':
        cart.add('A', 3)

    elif fixture_name == '300k':
        cart.add('B', 1)

    else:
        raise ValueError(
            f'UNKNOWN_FIXTURE: {fixture_name!r}'
        )

    return service, customer, cart


Writing ecommerce_app.py


In [4]:
import ecommerce_app, importlib
importlib.reload(ecommerce_app)
from ecommerce_app import fixture



In [5]:
%%writefile test_cases.csv
test_case_id,requirement_id,condition_id,scenario_id,title,priority,precondition,fixture_key,input_json,expected_type,expected_json,test_steps,execution_status,actual_result,error_message,evidence_reference
TC-02-001,REQ-02,COND-CART-01,TS-CART-01,Thêm số lượng 1 hợp lệ,P1,Fresh fixture; item and customer states defined by fixture,EMPTY_CART,"{""operation"": ""add"", ""sku"": ""A"", ""qty"": 1}",value,"{""qty"": 1, ""lines"": 1}",Create fixture → capture initial state → invoke target → assert expected and state → log,,,,
TC-02-002,REQ-02,COND-CART-01,TS-CART-01,Cộng dồn SKU trùng 2+3,P1,Fresh fixture; item and customer states defined by fixture,STANDARD,"{""operation"": ""add"", ""sku"": ""A"", ""qty"": 3}",value,"{""qty"": 5, ""lines"": 1}",Create fixture → capture initial state → invoke target → assert expected and state → log,,,,
TC-02-003,REQ-02,COND-CART-02,TS-CART-02,Từ chối số lượng 0,P0,Fresh fixture; item and customer states defined by fixture,EMPTY_CART,"{""operation"": ""add"", ""sku"": ""A"", ""qty"": 0}",exception,"{""error"": ""INVALID_QUANTITY"", ""qty"": 0, ""lines"": 0}",Create fixture → capture initial state → invoke target → assert expected and state → log,,,,
TC-02-004,REQ-02,COND-CART-02,TS-CART-02,Từ chối số lượng âm,P0,Fresh fixture; item and customer states defined by fixture,EMPTY_CART,"{""operation"": ""add"", ""sku"": ""A"", ""qty"": -1}",exception,"{""error"": ""INVALID_QUANTITY"", ""qty"": 0, ""lines"": 0}",Create fixture → capture initial state → invoke target → assert expected and state → log,,,,
TC-02-005,REQ-02,COND-CART-02,TS-CART-02,Từ chối giá trị Boolean làm số lượng sản phẩm,P0,"EMPTY_CART; giỏ rỗng, SKU A hợp lệ; quantity phải là số nguyên dương và không nhận Boolean",EMPTY_CART,"{""operation"": ""add"", ""sku"": ""A"", ""qty"": true}",error,"{""error"": ""INVALID_QUANTITY"", ""qty"": 0, ""lines"": 0}",Khởi tạo EMPTY_CART → thêm A với qty=true → đối chiếu INVALID_QUANTITY và giỏ không thay đổi,,,,
TC-04-001,REQ-04,COND-AUTH-01,TS-AUTH-01,Khách đăng nhập checkout COD,P1,Fresh fixture; item and customer states defined by fixture,STANDARD,"{""operation"": ""checkout"", ""method"": ""COD""}",value,"{""status"": ""PENDING_COD"", ""orders"": 1}",Create fixture → capture initial state → invoke target → assert expected and state → log,,,,
TC-04-002,REQ-04,COND-AUTH-02,TS-AUTH-02,Khách ẩn danh bị từ chối COD,P0,Fresh fixture; item and customer states defined by fixture,ANONYMOUS_USER,"{""operation"": ""checkout"", ""method"": ""COD""}",exception,"{""error"": ""LOGIN_REQUIRED"", ""orders"": 0, ""stock_unchanged"": true}",Create fixture → capture initial state → invoke target → assert expected and state → log,,,,
TC-04-003,REQ-04,COND-AUTH-02,TS-AUTH-02,Khách ẩn danh bị từ chối ONLINE,P0,Fresh fixture; item and customer states defined by fixture,ANONYMOUS_USER,"{""operation"": ""checkout"", ""method"": ""ONLINE""}",exception,"{""error"": ""LOGIN_REQUIRED"", ""orders"": 0, ""stock_unchanged"": true}",Create fixture → capture initial state → invoke target → assert expected and state → log,,,,
TC-09-001,REQ-09,COND-SHP-01,TS-SHP-01,Nội thành dưới ngưỡng 499999,P0,Fresh fixture; item and customer states defined by fixture,499999,"{""operation"": ""shipping"", ""area"": ""inner""}",value,"{""fee"": 30000}",Create fixture → capture initial state → invoke target → assert expected and state → log,,,,
TC-09-002,REQ-09,COND-SHP-01,TS-SHP-01,Nội thành đúng ngưỡng 500000,P0,Fresh fixture; item and customer states defined by fixture,500k,"{""operation"": ""shipping"", ""area"": ""inner""}",value,"{""fee"": 0}",Create fixture → capture initial state → invoke target → assert expected and state → log,,,,
TC-09-003,REQ-09,COND-SHP-01,TS-SHP-01,Nội thành trên ngưỡng 600000,P1,Fresh fixture; item and customer states defined by fixture,600k,"{""operation"": ""shipping"", ""area"": ""inner""}",value,"{""fee"": 0}",Create fixture → capture initial state → invoke target → assert expected and state → log,,,,
TC-09-004,REQ-09,COND-SHP-02,TS-SHP-02,Ngoại thành subtotal 300000,P1,Fresh fixture; item and customer states defined by fixture,300k,"{""operation"": ""shipping"", ""area"": ""outer""}",value,"{""fee"": 50000}",Create fixture → capture initial state → invoke target → assert expected and state → log,,,,
TC-09-005,REQ-09,COND-SHP-02,TS-SHP-02,Ngoại thành subtotal 500000,P1,Fresh fixture; item and customer states defined by fixture,500k,"{""operation"": ""shipping"", ""area"": ""outer""}",value,"{""fee"": 50000}",Create fixture → capture initial state → invoke target → assert expected and state → log,,,,
TC-10-001,REQ-10,COND-PAY-01,TS-PAY-01,COD trả PENDING_COD,P1,Fresh fixture; item and customer states defined by fixture,600k,"{""operation"": ""checkout"", ""method"": ""COD""}",value,"{""status"": ""PENDING_COD"", ""orders"": 1, ""charges"": 0}",Create fixture → capture initial state → invoke target → assert expected and state → log,,,,
TC-10-002,REQ-10,COND-PAY-02,TS-PAY-02,ONLINE approved trả PAID,P1,Fresh fixture; item and customer states defined by fixture,600k,"{""operation"": ""checkout"", ""method"": ""ONLINE"", ""approved"": true}",value,"{""status"": ""PAID"", ""orders"": 1, ""charges"": 1}",Create fixture → capture initial state → invoke target → assert expected and state → log,,,,
TC-10-003,REQ-10,COND-PAY-02,TS-PAY-02,ONLINE declined không tạo order,P0,Fresh fixture; item and customer states defined by fixture,600k,"{""operation"": ""checkout"", ""method"": ""ONLINE"", ""approved"": false}",exception,"{""error"": ""PAYMENT_DECLINED"", ""orders"": 0, ""charges"": 0}",Create fixture → capture initial state → invoke target → assert expected and state → log,,,,
TC-10-004,REQ-10,COND-PAY-02,TS-PAY-02,Từ chối phương thức thanh toán không được hỗ trợ,P0,Fixture 600k; khách hàng đã đăng nhập; giỏ hàng có hàng hợp lệ; kiểm tra chỉ nghiệp vụ Payment,600k,"{""operation"": ""checkout"", ""method"": ""WALLET""}",exception,"{""error"": ""INVALID_PAYMENT_METHOD"", ""orders"": 0, ""charges"": 0}","Khởi tạo fixture 600k → checkout với method WALLET → đối chiếu lỗi INVALID_PAYMENT_METHOD, số order và charge; không kiểm thử hoàn tồn kho",,,,
TC-12-001,REQ-12,COND-ORD-01,TS-ORD-01,Hai yêu cầu khác key có Order ID riêng,P0,Fresh fixture; item and customer states defined by fixture,STANDARD,"{""operation"": ""two_orders""}",value,"{""unique_ids"": true, ""orders"": 2}",Create fixture → capture initial state → invoke target → assert expected and state → log,,,,
TC-12-002,REQ-12,COND-ORD-02,TS-ORD-02,Order COD tính tổng 400000+30000,P1,Fresh fixture; item and customer states defined by fixture,STANDARD,"{""operation"": ""checkout"", ""method"": ""COD"", ""area"": ""inner""}",value,"{""total"": 430000, ""orders"": 1}",Create fixture → capture initial state → invoke target → assert expected and state → log,,,,
TC-12-003,REQ-12,COND-ORD-02,TS-ORD-02,Tổng tiền bằng 0 nếu discount vượt subtotal,P0,Fresh fixture; item and customer states defined by fixture,STANDARD,"{""operation"": ""calc_total"", ""subtotal"": 100000, ""discount"": 200000, ""shipping"": 0}",value,"{""total"": 0}",Create fixture → capture initial state → invoke target → assert expected and state → log,,,,
TC-12-004,REQ-12,COND-ORD-02,TS-ORD-02,Tổng tiền COD nội thành tại đúng ngưỡng 500.000 VND (không voucher),P0,Fixture 500k; khách hàng đã đăng nhập; giỏ hợp lệ; không dùng mã giảm giá,500k,"{""operation"": ""checkout"", ""method"": ""COD"", ""area"": ""inner""}",value,"{""total"": 500000, ""orders"": 1}","Khởi tạo 500k → checkout COD nội thành không voucher → xác minh total 500000, orders 1",,,,
TC-12-005,REQ-12,COND-ORD-02,TS-ORD-02,Tổng tiền COD ngoại thành khi subtotal 600.000 VND (không voucher),P1,Fixture 600k; khách hàng đã đăng nhập; giỏ hợp lệ; không dùng mã giảm giá,600k,"{""operation"": ""checkout"", ""method"": ""COD"", ""area"": ""outer""}",value,"{""total"": 650000, ""orders"": 1}","Khởi tạo 600k → checkout COD ngoại thành không voucher → xác minh total 650000, orders 1",,,,


Writing test_cases.csv


In [6]:
%%writefile test_runner.py

"""CSV-driven execution for STLab02. Source baseline remains intentionally unpatched."""

import csv,json,time,datetime,pathlib,collections,html
import pandas as pd
from decimal import Decimal
from ecommerce_app import fixture,Cart,D

ROOT=pathlib.Path(__file__).parent
REQUIRED={'test_case_id','requirement_id','condition_id','scenario_id','fixture_key','input_json','expected_type','expected_json'}


def run_one(row):

    started=time.perf_counter()
    actual={}
    err=''
    outcome='FAIL'

    try:
        request=json.loads(row['input_json'])
        expected=json.loads(row['expected_json'])

        if not isinstance(request,dict) or not isinstance(expected,dict):
            raise TypeError('JSON_OBJECT_REQUIRED')

        app,user,cart=fixture(row['fixture_key'])
        op=request['operation']

        before={k:v.stock for k,v in app.catalog.products.items()}

        try:

            if op=='add':
                cart.add(request['sku'],request['qty'])
                actual={'qty':cart.total_quantity(),'lines':len(cart.lines)}

            elif op=='shipping':
                actual={'fee':int(app.shipping.calculate(cart.subtotal(),request['area']))}

            elif op=='checkout':
                order=app.place(user,cart,area=request.get('area','inner'),voucher=request.get('voucher'),method=request.get('method','COD'),approved=request.get('approved',True),key=row['test_case_id'])
                actual={'status':order.status,'total':int(order.total)}

            elif op=='two_orders':
                a=app.place(user,cart,key='ONE')
                b=app.place(user,cart,key='TWO')
                actual={'unique_ids':a.order_id!=b.order_id}

            elif op=='calc_total':
                actual={'total':int(app.calculate_total(D(request['subtotal']),D(request['discount']),D(request['shipping'])))}

            else:
                raise RuntimeError('UNKNOWN_OPERATION:'+str(op))

        except ValueError as e:
            err=str(e)
            actual={'error':err}

        after={k:v.stock for k,v in app.catalog.products.items()}

        if op=='add':
            actual.update(qty=cart.total_quantity(),lines=len(cart.lines))

        actual.update(orders=len(app.orders),charges=len(app.payments.charges),stock_unchanged=before==after,stock_before=before,stock_after=after)

        mismatch={
            k:{'expected':v,'actual':actual.get(k,'<MISSING>')}
            for k,v in expected.items()
            if actual.get(k)!=v
        }

        # Expected ValueError must match exactly; unexpected exceptions always fail.

        if err and 'error' not in expected:
            mismatch['unexpected_exception']={'actual':err}

        outcome='PASS' if not mismatch else 'FAIL'

    except (KeyError,ValueError,TypeError,json.JSONDecodeError) as e:
        outcome='BLOCKED'
        err='SETUP_ERROR:'+str(e)
        mismatch={'setup_error':err}

    except Exception as e:
        outcome='BLOCKED'
        err='RUNNER_ERROR:'+repr(e)
        mismatch={'runner_error':err}

    return dict(row,execution_status=outcome,actual_result=json.dumps(actual,ensure_ascii=False),error_message=err,evidence_reference=f"reports/test_execution.csv#{row.get('test_case_id','')}",comparison=json.dumps(mismatch,ensure_ascii=False),execution_time_ms=round((time.perf_counter()-started)*1000,3),executed_at=datetime.datetime.now(datetime.timezone.utc).isoformat())


def main():

    with (ROOT/'test_cases.csv').open(encoding='utf-8-sig',newline='') as f:
        reader=csv.DictReader(f)
        missing=REQUIRED-set(reader.fieldnames or [])

        if missing:
            raise RuntimeError(f'MISSING_COLUMNS:{missing}')

        rows=list(reader)

    if len({r['test_case_id'] for r in rows})!=len(rows):
        raise RuntimeError('DUPLICATE_TEST_IDS')

    results=[run_one(r) for r in rows]

    if len(results)!=len(rows):
        raise RuntimeError('ROW_COUNT_MISMATCH')

    out=ROOT/'reports'
    out.mkdir(exist_ok=True)

    with (out/'test_execution.csv').open('w',encoding='utf-8-sig',newline='') as f:
        writer=csv.DictWriter(f,fieldnames=list(results[0]),extrasaction='ignore')
        writer.writeheader()
        writer.writerows(results)

    stats=collections.Counter(r['execution_status'] for r in results)
    byreq=collections.defaultdict(collections.Counter)

    for r in results:
        byreq[r['requirement_id']][r['execution_status']]+=1

    cards=''.join(
        f'<div class="card"><b>{html.escape(k)}</b><br>{v}</div>'
        for k,v in [('TOTAL',len(results)),('PASS',stats['PASS']),('FAIL',stats['FAIL']),('BLOCKED',stats['BLOCKED'])]
    )

    trs=''.join(
        '<tr>'+''.join(
            f'<td>{html.escape(str(r.get(col,"")))}</td>'
            for col in ['test_case_id','requirement_id','title','execution_status','expected_json','actual_result','comparison']
        )+'</tr>'
        for r in results
    )

    bars=''.join(
        f'<tr><td>{k}</td><td>{sum(v.values())}</td><td>{v["PASS"]}</td><td>{v["FAIL"]}</td><td>{v["BLOCKED"]}</td></tr>'
        for k,v in sorted(byreq.items())
    )

    page=f'''
<!doctype html>
<html lang="vi">

<head>
    <meta charset="utf-8">
    <title>STLab02 Execution Dashboard</title>

    <style>
        body{{font:15px Arial;margin:32px;color:#172c42;background:#f4f7fb}}
        h1{{color:#133b65}}
        .cards{{display:flex;gap:16px}}
        .card{{background:white;border-radius:8px;padding:22px;min-width:120px;box-shadow:0 1px 4px #ccc}}
        table{{width:100%;border-collapse:collapse;background:white;margin:20px 0;font-size:12px}}
        th,td{{border:1px solid #dce3ef;padding:8px;vertical-align:top;overflow-wrap:anywhere}}
        th{{background:#d7e6f6}}
    </style>
</head>

<body>
    <h1>EduCommerce — Test execution</h1>

    <p>
        Baseline: Instruction logic; expected independently from acceptance criteria.
        Generated UTC: {datetime.datetime.now(datetime.timezone.utc).isoformat()}
    </p>

    <section class="cards">
        {cards}
    </section>

    <h2>Results by requirement</h2>

    <table>
        <tr>
            <th>REQ</th>
            <th>TOTAL</th>
            <th>PASS</th>
            <th>FAIL</th>
            <th>BLOCKED</th>
        </tr>
        {bars}
    </table>

    <h2>Detailed execution</h2>

    <table>
        <tr>
            {''.join('<th>'+x+'</th>' for x in ['Case','REQ','Title','Status','Expected','Actual','Differences'])}
        </tr>
        {trs}
    </table>
</body>

</html>
'''

    (out/'test_dashboard.html').write_text(page,encoding='utf-8')

    df=pd.DataFrame(results)

    matrix=df[['requirement_id','condition_id','scenario_id','test_case_id','execution_status','evidence_reference','executed_at']].copy()

    summary_rows=[
        {'requirement_id':k,'total':sum(v.values()),'PASS':v['PASS'],'FAIL':v['FAIL'],'BLOCKED':v['BLOCKED']}
        for k,v in sorted(byreq.items())
    ]

    totals=pd.DataFrame([
        {'total':len(results),'PASS':stats['PASS'],'FAIL':stats['FAIL'],'BLOCKED':stats['BLOCKED'],'pass_rate_pct':round(100*stats['PASS']/len(results),2) if results else 0}
    ])

    with pd.ExcelWriter(out/'test_report.xlsx',engine='openpyxl') as xls:
        df.to_excel(xls,sheet_name='Execution',index=False)
        pd.DataFrame(summary_rows).to_excel(xls,sheet_name='By Requirement',index=False)
        matrix.to_excel(xls,sheet_name='Traceability',index=False)
        totals.to_excel(xls,sheet_name='Summary',index=False)

    print('TOTAL',len(results),'PASS',stats['PASS'],'FAIL',stats['FAIL'],'BLOCKED',stats['BLOCKED'])

    for r in results:

        if r['execution_status']!='PASS':
            print(r['test_case_id'],r['execution_status'],r['comparison'])


if __name__=='__main__':
    main()


Writing test_runner.py


In [9]:
from pathlib import Path
import csv
import importlib, test_runner
importlib.reload(test_runner)
with open('test_cases.csv', encoding='utf-8-sig', newline='') as f:
    data = list(csv.DictReader(f))
print('CSV rows:', len(data))
print('Requirements:', sorted(set(x['requirement_id'] for x in data)))
print('Unique IDs:', len(set(x['test_case_id'] for x in data)))
assert len(data) == 22
assert set(x['requirement_id'] for x in data) == {'REQ-02','REQ-04','REQ-09','REQ-10','REQ-12'}


CSV rows: 22
Requirements: ['REQ-02', 'REQ-04', 'REQ-09', 'REQ-10', 'REQ-12']
Unique IDs: 22


In [10]:
!pip -q install pandas openpyxl matplotlib gradio
!python test_runner.py
import pandas as pd
results = pd.read_csv('reports/test_execution.csv').fillna('')
report_dir = Path('reports')
print(results[['test_case_id','requirement_id','execution_status','comparison']].to_string(index=False))


TOTAL 22 PASS 21 FAIL 1 BLOCKED 0
TC-02-005 FAIL {"error": {"expected": "INVALID_QUANTITY", "actual": "<MISSING>"}, "qty": {"expected": 0, "actual": 1}, "lines": {"expected": 0, "actual": 1}}
test_case_id requirement_id execution_status                                                                                                                                     comparison
   TC-02-001         REQ-02             PASS                                                                                                                                             {}
   TC-02-002         REQ-02             PASS                                                                                                                                             {}
   TC-02-003         REQ-02             PASS                                                                                                                                             {}
   TC-02-004         REQ-02             PASS            

In [12]:
summary = results.groupby(['requirement_id','execution_status']).size().unstack(fill_value=0)
print(summary)
print('Report folder:', report_dir.resolve())
print('Created:', sorted(str(p) for p in report_dir.iterdir()))
assert len(results) == 22
assert all(results['execution_status'].isin(['PASS','FAIL','BLOCKED']))


execution_status  FAIL  PASS
requirement_id              
REQ-02               1     4
REQ-04               0     3
REQ-09               0     5
REQ-10               0     4
REQ-12               0     5
Report folder: /content/reports
Created: ['reports/test_dashboard.html', 'reports/test_execution.csv', 'reports/test_report.xlsx']


In [13]:
import gradio as gr
import matplotlib.pyplot as plt
import pandas as pd

report_df = pd.read_csv('reports/test_execution.csv').fillna('')

def dashboard_view(req='ALL', status='ALL'):
    df = report_df.copy()
    if req != 'ALL': df = df[df['requirement_id'] == req]
    if status != 'ALL': df = df[df['execution_status'] == status]
    summary = df['execution_status'].value_counts().reindex(['PASS','FAIL','BLOCKED'], fill_value=0)
    fig, ax = plt.subplots(figsize=(6,4))
    bars = ax.bar(summary.index, summary.values)
    ax.bar_label(bars, padding=3)
    ax.set_title('Test execution results')
    ax.set_ylabel('Number of test cases')
    ax.set_ylim(0, max(2, int(summary.max())+2))
    fig.tight_layout()
    cols=['test_case_id','requirement_id','scenario_id','expected_json','actual_result','execution_status']
    return fig, df[cols], f'Executed: {len(df)} | PASS: {summary["PASS"]} | FAIL: {summary["FAIL"]} | BLOCKED: {summary["BLOCKED"]}'

with gr.Blocks(title='EduCommerce - Test Dashboard') as demo:
    gr.Markdown('# EduCommerce — Requirement Traceability & Test Results')
    with gr.Row():
        req_filter=gr.Dropdown(['ALL']+sorted(report_df.requirement_id.unique().tolist()), value='ALL',label='Requirement')
        status_filter=gr.Dropdown(['ALL','PASS','FAIL','BLOCKED'],value='ALL',label='Test status')
    status_line=gr.Markdown()
    with gr.Row(equal_height=True):
        with gr.Column(scale=1,min_width=300):
            chart=gr.Plot(label='Result distribution')
        with gr.Column(scale=2,min_width=420):
            result_table=gr.Dataframe(label='Test execution details',interactive=False,wrap=True)
    gr.File(value='reports/test_report.xlsx',label='Download Excel report')
    req_filter.change(dashboard_view,[req_filter,status_filter],[chart,result_table,status_line])
    status_filter.change(dashboard_view,[req_filter,status_filter],[chart,result_table,status_line])
    demo.load(dashboard_view,[req_filter,status_filter],[chart,result_table,status_line])
# Can use share=True to publish a temporary external URL; default is local runtime.
demo.launch(share=False,debug=False)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

In [14]:
from google.colab import files
files.download('reports/test_report.xlsx')
# Optionally download CSV / dashboard:
# files.download('reports/test_execution.csv')
# files.download('reports/test_dashboard.html')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>